# Encodage Catégoriel
Entrée : `metadata_features.parquet` → Sortie : `features_encodees.parquet`

- Suppression des colonnes string constantes, redondantes ou à trop haute cardinalité
- One-hot encoding des colonnes catégorielles utiles (drop_first=True)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_features.parquet')
df_enc = df.copy()

print('Dataset chargé :', df_enc.shape)

Dataset chargé : (549971, 395)


## 1. Suppression des colonnes string inutiles

In [ ]:
DROP_REDUNDANT = [
    'in.building_america_climate_zone',
    'in.cec_climate_zone',
    'in.energystar_climate_zone_2023',
    'in.ashrae_iecc_climate_zone_2004_sub_cz_split',
    'in.hvac_heating_type_and_fuel',
    'in.census_division',
    'in.census_division_recs',
    'in.geometry_building_type_acs',
    'in.location_region',
    'in.ahs_region',
    'in.iso_rto_region',
    'in.puma_metro_status',
    'in.county_metro_status',
    'in.generation_and_emissions_assessment_region',
    'in.metropolitan_and_micropolitan_statistical_area',
    'in.reeds_balancing_area',
    'in.county_and_puma',
    'in.county_name',
    'in.county',
    'in.puma',
    'in.city',
    'in.state',
    'in.weather_file_city',
    'in.heating_unavailable_period',
    'in.cooling_unavailable_period',
    'in.upgrade_name',
]
# On peut ajouter ici peut etre les colonnes : 'in.utility_bill_fuel_oil_fixed_charges','in.utility_bill_propane_fixed_charges' Car contient que des zeros(mais aussi on peut pas supimer cette informations)
str_cols = df_enc.select_dtypes(exclude=[np.number]).columns
drop_const    = [c for c in str_cols if df_enc[c].nunique() <= 1]
drop_explicit = [c for c in DROP_REDUNDANT if c in df_enc.columns]

to_drop = list(set(drop_const + drop_explicit))
df_enc.drop(columns=to_drop, inplace=True)

print(f'Colonnes constantes supprimées  : {len(drop_const)}')
print(f'Colonnes redondantes supprimées : {len(drop_explicit)}')
print(f'Shape : {df_enc.shape}')

Colonnes constantes supprimées  : 21
Colonnes redondantes supprimées : 26
Shape : (549971, 349)


## 2. Imputation par groupe (zone climatique)
Doit s'exécuter ici — avant le OHE — pendant que la zone climatique est encore une chaîne.

In [3]:
GROUPBY_COL = 'in.ashrae_iecc_climate_zone_2004'

num_cols = df_enc.select_dtypes(include=[np.number]).columns
nan_cols = [c for c in num_cols if df_enc[c].isna().any()]

if nan_cols and GROUPBY_COL in df_enc.columns:
    for c in nan_cols:
        global_med  = df_enc[c].median()
        group_meds  = df_enc.groupby(GROUPBY_COL)[c].transform('median')
        df_enc[c]   = df_enc[c].fillna(group_meds.fillna(global_med))
    print(f'Imputation par zone climatique : {len(nan_cols)} colonnes')
    print(f'NaN résiduels : {df_enc[nan_cols].isna().sum().sum()}')
else:
    print('Colonne de groupement absente — ignoré.')

Imputation par zone climatique : 30 colonnes
NaN résiduels : 0


## 3. Groupe 1 — Zone climatique
`in.ashrae_iecc_climate_zone_2004` — 17 zones (1A … 8AK)

In [4]:
col = 'in.ashrae_iecc_climate_zone_2004'

df_enc[col] = df_enc[col].astype('category')
df_enc = pd.get_dummies(df_enc, columns=[col], prefix='climate', dtype=np.int8, drop_first=True)

print(f'Shape : {df_enc.shape}')

Shape : (549971, 364)


## 4. Groupe 2 — Géométrie

| Colonne | Stratégie |
|---|---|
| `in.orientation` | sin/cos circulaire (8 directions) |
| `in.window_areas` | extraction numérique F/B/L/R (%) |
| `in.geometry_building_number_units_mf/sfa` | midpoint numérique |
| `in.geometry_building_level_mf` | ordinal (Bottom=0, Middle=1, Top=2) |
| `in.geometry_building_horizontal_location_mf/sfa` | OHE drop_first |
| `in.geometry_attic_type` | ordinal thermique (Not Applicable=0 → Conditioned=3) |
| `in.geometry_foundation_type` | ordinal thermique (Ambient=0 → Heated Basement=4) |
| `in.geometry_garage` | parse → `garage_cars` (0-3) + `garage_conditioned` (0/1) |
| `in.neighbors` | ordinal (None=0, un côté=1, Both=2) |
| `in.door_area` | numérique ft² → m² |
| `in.corridor` | supprimé (tout NaN après transformations_numeriques) |
| `in.geometry_wall_type` | OHE drop_first |
| `in.geometry_wall_exterior_finish` | OHE drop_first |
| `in.geometry_building_type_recs` | OHE drop_first |

In [ ]:
import re

# ── Orientation → sin/cos ────────────────────────────────────────────────────
ORIENT_DEG = {
    'North': 0, 'Northeast': 45, 'East': 90, 'Southeast': 135,
    'South': 180, 'Southwest': 225, 'West': 270, 'Northwest': 315,
}
col = 'in.orientation'
if col in df_enc.columns:
    deg = df_enc[col].map(ORIENT_DEG)
    df_enc['orientation_sin'] = np.sin(np.radians(deg)).astype('float32')
    df_enc['orientation_cos'] = np.cos(np.radians(deg)).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('orientation → sin/cos OK')

# ── Window areas → F/B/L/R numériques ───────────────────────────────────────
def parse_window_areas(val):
    if pd.isna(val):
        return np.nan, np.nan, np.nan, np.nan
    parts = re.findall(r'([FBLR])(\d+)', str(val).upper())
    d = {k: float(v) for k, v in parts}
    return d.get('F', np.nan), d.get('B', np.nan), d.get('L', np.nan), d.get('R', np.nan)

col = 'in.window_areas'
if col in df_enc.columns:
    parsed = df_enc[col].apply(lambda v: pd.Series(parse_window_areas(v),
                                index=['window_front', 'window_back', 'window_left', 'window_right']))
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('window_areas → 4 colonnes OK')

# ── Number of units → colonne fusionnée mf + sfa ────────────────────────────
def midpoint_units(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ('None', 'nan'): return np.nan
    if s.endswith('+'): return float(s[:-1])
    m = re.match(r'(\d+)-(\d+)', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2
    return pd.to_numeric(s, errors='coerce')

col_mf  = 'in.geometry_building_number_units_mf'
col_sfa = 'in.geometry_building_number_units_sfa'
if col_mf in df_enc.columns or col_sfa in df_enc.columns:
    mf  = df_enc[col_mf].apply(midpoint_units)  if col_mf  in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    sfa = df_enc[col_sfa].apply(midpoint_units) if col_sfa in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    df_enc['building_number_units'] = mf.combine_first(sfa).astype('float32')
    df_enc.drop(columns=[c for c in [col_mf, col_sfa] if c in df_enc.columns], inplace=True)
    print('building_number_units (mf + sfa fusionnés) OK')

# ── Building level → ordinal ─────────────────────────────────────────────────
LEVEL_MAP = {'Bottom': 0, 'Middle': 1, 'Top': 2}
col = 'in.geometry_building_level_mf'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(LEVEL_MAP).astype('Int8')
    print('geometry_building_level_mf → ordinal OK')

# ── Horizontal location → colonne fusionnée mf + sfa → OHE ──────────────────
col_mf  = 'in.geometry_building_horizontal_location_mf'
col_sfa = 'in.geometry_building_horizontal_location_sfa'
if col_mf in df_enc.columns or col_sfa in df_enc.columns:
    loc_mf  = df_enc[col_mf].replace({'None': np.nan, 'Not Applicable': np.nan}) if col_mf  in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    loc_sfa = df_enc[col_sfa].replace({'None': np.nan})                           if col_sfa in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    df_enc['building_horizontal_location'] = loc_mf.combine_first(loc_sfa)
    df_enc.drop(columns=[c for c in [col_mf, col_sfa] if c in df_enc.columns], inplace=True)
    df_enc['building_horizontal_location'] = df_enc['building_horizontal_location'].astype('category')
    df_enc = pd.get_dummies(df_enc, columns=['building_horizontal_location'],
                            prefix='horiz_loc', dtype=np.int8, drop_first=True)
    print('building_horizontal_location (mf + sfa fusionnés) → OHE OK')

print(f'\nShape : {df_enc.shape}')

In [ ]:
import re

# ── sqft..ft2 → suppression (redondant avec geometry_floor_area) ─────────────
if 'in.sqft..ft2' in df_enc.columns:
    df_enc.drop(columns=['in.sqft..ft2'], inplace=True)
    print('sqft..ft2 supprimé (redondant avec geometry_floor_area)')

# ── door_area → suppression (colonne constante : "20 ft^2") ──────────────────
if 'in.door_area' in df_enc.columns:
    df_enc.drop(columns=['in.door_area'], inplace=True)
    print('door_area supprimé (constante)')

# ── Attic type → ordinal (qualité thermique croissante) ─────────────────────
ATTIC_MAP = {
    'None':                                 0,  # pas de combles
    'Vented Attic':                         1,  # combles ventilés (non conditionnés)
    'Unvented Attic':                       2,  # combles non ventilés (mieux isolés)
    'Finished Attic or Cathedral Ceilings': 3,  # combles aménagés/conditionnés
}
col = 'in.geometry_attic_type'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(ATTIC_MAP).astype('Int8')
    print(f'attic_type → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Foundation type → ordinal (gradient thermique sol) ──────────────────────
FOUNDATION_MAP = {
    'Ambient':             0,
    'Slab':                1,
    'Vented Crawlspace':   2,
    'Unvented Crawlspace': 2,
    'Unheated Basement':   3,
    'Heated Basement':     4,
}
col = 'in.geometry_foundation_type'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(FOUNDATION_MAP).astype('Int8')
    print(f'foundation_type → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Garage → ordinal (nombre de voitures) ───────────────────────────────────
GARAGE_MAP = {'None': 0, '1 Car': 1, '2 Car': 2, '3 Car': 3}
col = 'in.geometry_garage'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(GARAGE_MAP).astype('Int8')
    print(f'garage → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Neighbors → distance (ft) + flag both sides ──────────────────────────────
def parse_neighbors(val):
    if pd.isna(val): return np.nan, 0
    s = str(val).strip()
    if s == 'None': return 0.0, 0
    m = re.search(r'(\d+)', s)
    dist = float(m.group(1)) if m else np.nan
    both = int('Left/Right' in s or 'Both' in s)
    return dist, both

col = 'in.neighbors'
if col in df_enc.columns:
    parsed = df_enc[col].apply(
        lambda v: pd.Series(parse_neighbors(v), index=['neighbor_distance_ft', 'neighbor_both_sides'])
    )
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('neighbors → neighbor_distance_ft + neighbor_both_sides OK')

# ── Corridor → binaire ───────────────────────────────────────────────────────
CORRIDOR_MAP = {'Not Applicable': 0, 'Double-Loaded Interior': 1}
col = 'in.corridor'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(CORRIDOR_MAP).fillna(0).astype('Int8')
    print(f'corridor → binaire OK  {df_enc[col].value_counts().to_dict()}')

# ── Wall exterior finish → matériau (OHE) + réflectivité (binaire) ───────────
FINISH_MATERIAL = {
    'Vinyl, Light':                 'vinyl',
    'Aluminum, Light':              'aluminum',
    'Wood, Medium/Dark':            'wood',
    'Brick, Light':                 'brick',
    'Brick, Medium/Dark':           'brick',
    'Stucco, Light':                'stucco',
    'Stucco, Medium/Dark':          'stucco',
    'Fiber-Cement, Light':          'fiber_cement',
    'Shingle, Composition, Medium': 'shingle',
    'Shingle, Asbestos, Medium':    'shingle',
    'None':                         'none',
}
col = 'in.geometry_wall_exterior_finish'
if col in df_enc.columns:
    df_enc['wall_finish_material'] = df_enc[col].map(FINISH_MATERIAL).astype('category')
    df_enc['wall_finish_dark']     = df_enc[col].str.contains('Medium|Dark', na=False).astype(np.int8)
    df_enc.drop(columns=[col], inplace=True)
    df_enc = pd.get_dummies(df_enc, columns=['wall_finish_material'],
                            prefix='wall_finish', dtype=np.int8, drop_first=True)
    print('wall_exterior_finish → wall_finish_<materiau> + wall_finish_dark OK')

# ── Wall type + building type RECS → OHE ─────────────────────────────────────
OHE_COLS = {
    'in.geometry_wall_type':          'wall_type',
    'in.geometry_building_type_recs': 'bldg_type',
}
ohe_present = {k: v for k, v in OHE_COLS.items() if k in df_enc.columns}
if ohe_present:
    for c in ohe_present:
        df_enc[c] = df_enc[c].astype('category')
    df_enc = pd.get_dummies(df_enc, columns=list(ohe_present.keys()),
                            prefix=list(ohe_present.values()),
                            dtype=np.int8, drop_first=True)
    print(f'OHE : {list(ohe_present.keys())}')

print(f'\nShape : {df_enc.shape}')

## 5. Groupe 3 — Enveloppe thermique

| Colonne | Stratégie |
|---|---|
| `in.insulation_wall/ceiling/roof/floor/foundation_wall/rim_joist/slab` | déjà R-values numériques (`transformations_numeriques`) |
| `in.air_leakage_to_outside_ach50` | déjà float |
| `in.infiltration` | supprimé (redondant avec `air_leakage_to_outside_ach50`) |
| `in.interior_shading` / `in.overhangs` / `in.eaves` / `in.doors` / `in.radiant_barrier` | supprimés (colonnes constantes) |
| `in.windows` | décomposé → `window_panes` (1/2/3) + `window_low_e` + `window_metal_frame` + `window_storm` |
| `in.ground_thermal_conductivity` | numérique float (W/m·K) |
| `in.roof_material` | à traiter (collègue) |

In [ ]:
# ── Colonnes constantes + redondantes → suppression ──────────────────────────
DROP_ENVELOPE = [
    'in.interior_shading',  # constante : "Summer = 0.7, Winter = 0.85"
    'in.overhangs',         # constante : "None"
    'in.eaves',             # constante : "2 ft"
    'in.doors',             # constante : "Fiberglass"
    'in.radiant_barrier',   # constante : "No" / "None"
    'in.infiltration',      # redondant avec air_leakage_to_outside_ach50
]
dropped = [c for c in DROP_ENVELOPE if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'Supprimés ({len(dropped)}) : {dropped}')

# ── Windows → 4 features physiques ───────────────────────────────────────────
def parse_windows(val):
    if pd.isna(val):
        return np.nan, np.nan, np.nan, np.nan
    s = str(val)
    panes      = 1 if s.startswith('Single') else (3 if s.startswith('Triple') else 2)
    low_e      = int('Low-E' in s)
    metal      = int('Metal' in s and 'Non-metal' not in s)
    storm      = int('Storm' in s)
    return panes, low_e, metal, storm

col = 'in.windows'
if col in df_enc.columns:
    parsed = df_enc[col].apply(
        lambda v: pd.Series(parse_windows(v),
                            index=['window_panes', 'window_low_e', 'window_metal_frame', 'window_storm'])
    )
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('Int8')], axis=1)
    print('windows → window_panes + window_low_e + window_metal_frame + window_storm OK')

# ── Ground thermal conductivity → float ──────────────────────────────────────
col = 'in.ground_thermal_conductivity'
if col in df_enc.columns:
    df_enc[col] = pd.to_numeric(df_enc[col], errors='coerce').astype('float32')
    print(f'ground_thermal_conductivity → float OK  {sorted(df_enc[col].dropna().unique())}')

print(f'\nShape : {df_enc.shape}')

## 6. Groupe 3 (suite) — Roof material
Encodage par R-value thermique (isolation thermique du matériau de toiture)

In [ ]:
#https://roofobservations.com/r-value-table-building-materials/#google_vignette
#https://www.cupapizarras.com/uk/news/thermal-behavior-roofing-slates/
#https://web.ornl.gov/sci/buildings/conf-archive/2013%20B12%20papers/163-Olson.pdf
#https://ruggedcoatings.com/understanding-r-value-in-roofing-systems/
#https://lorisweb.com/CMGT235/DIS02/R-Values%20Lookup.pdf

def roof_type(rf_str):
    if pd.isna(rf_str):
        return np.nan
    s = str(rf_str).strip()
    if 'Asphalt Shingles, Medium' in s:
        return 0.44
    elif 'Composition Shingles' in s:
        return 0.44
    elif 'Metal, Dark' in s:
        return 0.17
    elif 'Slate' in s:
        return 0.05
    elif 'Tile, Clay or Ceramic' in s:
        return 0.80
    elif 'Tile, Concrete' in s:
        return 0.60
    elif 'Wood Shingles' in s:
        return 1.00
    return np.nan

col = 'in.roof_material'
if col in df_enc.columns:
    df_enc['roof_material'] = df_enc[col].apply(roof_type).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print(f'roof_material → R-value OK  {df_enc["roof_material"].value_counts().sort_index().to_dict()}')

print(f'\nShape : {df_enc.shape}')

## 5. Groupe 3 — Systèmes HVAC
`in.hvac_heating_type`, `in.hvac_cooling_type`, `in.hvac_heating_fuel` — OHE drop_first

In [6]:
HVAC_COLS = [
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.hvac_heating_fuel',
]

hvac_cols_present = [c for c in HVAC_COLS if c in df_enc.columns]

for c in hvac_cols_present:
    df_enc[c] = df_enc[c].astype('category')

df_enc = pd.get_dummies(df_enc, columns=hvac_cols_present,
                        prefix=[c.split('in.')[-1] for c in hvac_cols_present],
                        dtype=np.int8, drop_first=True)

print(f'Colonnes HVAC encodées : {hvac_cols_present}')
print(f'Shape : {df_enc.shape}')

Colonnes HVAC encodées : ['in.hvac_heating_type', 'in.hvac_cooling_type']
Shape : (549971, 379)


In [ ]:
# ── Heating efficiency → séparer AFUE (fuel/élec) et HSPF (PAC) ──────────────
# Problème : la colonne mélange deux métriques incomparables
#   AFUE ∈ [0.60, 1.0]  → combustible (gaz, fuel) + électrique résistance
#   HSPF ∈ [6.2,  8.5]  → pompes à chaleur air-air (ASHP)
# Séparation par seuil : HSPF > 1.0, AFUE ≤ 1.0
# AFUE = 1.0 (électrique résistance) → NaN : constant par type, déjà encodé dans hvac_heating_type

col = 'in.hvac_heating_efficiency'
if col in df_enc.columns:
    eff = df_enc[col]
    df_enc['hvac_heating_eff_hspf'] = eff.where(eff > 1.0).astype('float32')
    df_enc['hvac_heating_eff_afue'] = eff.where(eff < 1.0).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('hvac_heating_efficiency → 2 colonnes :')
    print(f'  hvac_heating_eff_afue : {sorted(df_enc["hvac_heating_eff_afue"].dropna().unique())}')
    print(f'  hvac_heating_eff_hspf : {sorted(df_enc["hvac_heating_eff_hspf"].dropna().unique())}')

# ── Cooling efficiency → SEER/EER, même ordre de grandeur → une seule colonne ─
# NaN pour les PAC (leur efficacité chauffage est dans hvac_heating_eff_hspf)
col = 'in.hvac_cooling_efficiency'
if col in df_enc.columns:
    df_enc[col] = pd.to_numeric(df_enc[col], errors='coerce').astype('float32')
    print(f'hvac_cooling_efficiency → SEER/EER float OK  {sorted(df_enc[col].dropna().unique())}')

print(f'\nShape : {df_enc.shape}')

## 6. Groupe 4 — Période d'offset consignes (encodage circulaire)

| Colonne | Stratégie |
|---|---|
| `in.cooling_setpoint_offset_period` | direction + period_type + sin/cos heure réelle |
| `in.heating_setpoint_offset_period` | direction + period_type + sin/cos heure réelle |

In [7]:
DAY_START   = 9
NIGHT_START = 22

def encode_offset_period(val, default_start, with_direction=True):
    if pd.isna(val) or str(val).strip().lower() == 'none':
        base = [0, 0.0, 1.0]
        return [0] + base if with_direction else base

    s = str(val).strip().lower()

    period_type = int('day' in s) + 2 * int('night' in s)
    direction   = int('setup' in s) - int('setback' in s)

    m = re.search(r'([+-]\d+)h', s)
    shift = int(m.group(1)) if m else 0
    actual_hour = (default_start + shift) % 24

    sin_h = float(np.sin(2 * np.pi * actual_hour / 24))
    cos_h = float(np.cos(2 * np.pi * actual_hour / 24))

    base = [period_type, sin_h, cos_h]
    return [direction] + base if with_direction else base


PERIOD_COLS = {
    'in.cooling_setpoint_offset_period': ('cool', DAY_START,   True),
    'in.heating_setpoint_offset_period': ('heat', NIGHT_START, False),
}

for col, (prefix, default_start, with_dir) in PERIOD_COLS.items():
    if col in df_enc.columns:
        names = ([f'{prefix}_direction'] if with_dir else []) + [
            f'{prefix}_period_type', f'{prefix}_start_sin', f'{prefix}_start_cos'
        ]
        encoded = df_enc[col].apply(
            lambda v, ds=default_start, wd=with_dir: pd.Series(
                encode_offset_period(v, ds, wd), index=names
            )
        )
        df_enc = pd.concat([df_enc.drop(columns=[col]), encoded.astype('float32')], axis=1)
        print(f'{col} -> {names}')

print(f'\nShape : {df_enc.shape}')

in.cooling_setpoint_offset_period -> ['cool_direction', 'cool_period_type', 'cool_start_sin', 'cool_start_cos']
in.heating_setpoint_offset_period -> ['heat_period_type', 'heat_start_sin', 'heat_start_cos']

Shape : (549971, 384)


## 7. Groupe 4 — HVAC (suite)

| Colonne | Stratégie |
|---|---|
| `in.hvac_heating_type` / `in.hvac_cooling_type` | OHE (section précédente) |
| `in.hvac_heating_efficiency` / `in.hvac_cooling_efficiency` | déjà float (`transformations_numeriques`) |
| `in.hvac_has_ducts` / `in.hvac_has_zonal_electric_heating` / `in.heating_setpoint_has_offset` / `in.cooling_setpoint_has_offset` | déjà Int8 binaire (`transformations_numeriques`) |
| `in.hvac_heating_autosizing_factor` / `in.hvac_cooling_autosizing_factor` / `in.hvac_system_is_faulted` / `in.hvac_system_is_scaled` / `in.hvac_system_single_speed_*` / `in.mechanical_ventilation` / `in.dehumidifier` / `in.natural_ventilation` | supprimés (colonnes constantes) |
| `in.hvac_cooling_partial_space_conditioning` | % float (0.0 – 1.0) |
| `in.hvac_secondary_heating_partial_space_conditioning` | % float (0.0 – 1.0) |
| `in.hvac_secondary_heating_efficiency` | AFUE float |
| `in.heating_fuel` / `in.hvac_has_shared_system` / `in.hvac_secondary_heating_type` / `in.hvac_secondary_heating_fuel` / `in.hvac_shared_efficiencies` | OHE drop_first |

In [ ]:
import re

# ── Colonnes constantes → suppression ────────────────────────────────────────
DROP_HVAC_CONST = [
    'in.hvac_heating_autosizing_factor',
    'in.hvac_cooling_autosizing_factor',
    'in.hvac_system_is_faulted',
    'in.hvac_system_is_scaled',
    'in.hvac_system_single_speed_ac_airflow',
    'in.hvac_system_single_speed_ac_charge',
    'in.hvac_system_single_speed_ashp_airflow',
    'in.hvac_system_single_speed_ashp_charge',
    'in.mechanical_ventilation',
    'in.dehumidifier',
    'in.natural_ventilation',
]
dropped = [c for c in DROP_HVAC_CONST if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'Supprimés ({len(dropped)}) : {dropped}')

# ── Partial space conditioning → % float ─────────────────────────────────────
def parse_pct_conditioned(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.lower() == 'none': return 0.0
    m = re.search(r'(\d+)', s)
    return float(m.group(1)) / 100 if m else np.nan

for col in ['in.hvac_cooling_partial_space_conditioning',
            'in.hvac_secondary_heating_partial_space_conditioning']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].apply(parse_pct_conditioned).astype('float32')
        print(f'{col} → % float OK  {sorted(df_enc[col].dropna().unique())}')

# ── Secondary heating efficiency → AFUE float ────────────────────────────────
def parse_afue(val):
    if pd.isna(val) or str(val).strip().lower() == 'none': return np.nan
    m = re.search(r'(\d+\.?\d*)%', str(val))
    return float(m.group(1)) / 100 if m else np.nan

col = 'in.hvac_secondary_heating_efficiency'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].apply(parse_afue).astype('float32')
    print(f'secondary_heating_efficiency → AFUE float OK  {sorted(df_enc[col].dropna().unique())}')

# ── OHE ───────────────────────────────────────────────────────────────────────
OHE_HVAC = {
    'in.heating_fuel':                   'heat_fuel',
    'in.hvac_has_shared_system':         'hvac_shared',
    'in.hvac_secondary_heating_type':    'sec_heat_type',
    'in.hvac_secondary_heating_fuel':    'sec_heat_fuel',
    'in.hvac_shared_efficiencies':       'hvac_shared_eff',
}
ohe_present = {k: v for k, v in OHE_HVAC.items() if k in df_enc.columns}
if ohe_present:
    for c in ohe_present:
        df_enc[c] = df_enc[c].astype('category')
    df_enc = pd.get_dummies(df_enc,
                            columns=list(ohe_present.keys()),
                            prefix=list(ohe_present.values()),
                            dtype=np.int8, drop_first=True)
    print(f'OHE HVAC : {list(ohe_present.keys())}')

print(f'\nShape : {df_enc.shape}')

## 8. Groupe 5 — Occupants et socio-économique

| Colonne | Stratégie |
|---|---|
| `in.bedrooms` / `in.occupants` / `in.income` / `in.vintage` | déjà numériques (`transformations_numeriques`) |
| `in.units_represented` | supprimé (constante : `'1'`) |
| `in.representative_income` | supprimé (poids d'échantillonnage, pas une feature) |
| `in.tenure` | binaire (Owner=1, Renter=0, Not Available=NaN) |
| `in.vacancy_status` | binaire (Occupied=1, Vacant=0) |
| `in.household_has_tribal_persons` | binaire (Yes=1, No=0, Not Available=NaN) |
| `in.usage_level` | ordinal (Low=0, Medium=1, High=2) |
| `in.federal_poverty_level` | midpoint % → float (ex: `'100-150%'` → 1.25) |
| `in.area_median_income` | midpoint % → float |
| `in.state_metro_median_income` | midpoint % → float |

In [ ]:
import re

# ── Constantes + poids d'échantillonnage → suppression ───────────────────────
DROP_SOCIO = ['in.units_represented', 'in.representative_income']
dropped = [c for c in DROP_SOCIO if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'Supprimés : {dropped}')

# ── Binaires ─────────────────────────────────────────────────────────────────
BINARY_SOCIO = {
    'in.tenure':                       {'Owner': 1, 'Renter': 0},
    'in.vacancy_status':               {'Occupied': 1, 'Vacant': 0},
    'in.household_has_tribal_persons': {'Yes': 1, 'No': 0},
}
for col, mapping in BINARY_SOCIO.items():
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map(mapping).astype('Int8')
        print(f'{col.replace("in.", "")} → binaire OK  {df_enc[col].value_counts().to_dict()}')

# ── Usage level → ordinal ─────────────────────────────────────────────────────
col = 'in.usage_level'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map({'Low': 0, 'Medium': 1, 'High': 2}).astype('Int8')
    print(f'usage_level → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Revenus relatifs → midpoint % float ──────────────────────────────────────
# '0-100%' → 0.50  |  '100-150%' → 1.25  |  '400%+' → 4.0  |  'Not Available' → NaN
def parse_pct_range(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s == 'Not Available': return np.nan
    if s.endswith('%+'):
        return float(s[:-2]) / 100
    m = re.match(r'(\d+)-(\d+)%', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2 / 100
    m2 = re.match(r'(\d+)%', s)
    if m2: return float(m2.group(1)) / 100
    return np.nan

INCOME_PCT_COLS = [
    'in.federal_poverty_level',
    'in.area_median_income',
    'in.state_metro_median_income',
]
for col in INCOME_PCT_COLS:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].apply(parse_pct_range).astype('float32')
        print(f'{col.replace("in.", "")} → % float OK  {sorted(df_enc[col].dropna().unique())}')

print(f'\nShape : {df_enc.shape}')

## 9. Groupe 6 — Véhicule électrique

| Colonne | Stratégie |
|---|---|
| `in.electric_vehicle_ownership` / `in.electric_vehicle_outlet_access` | binaire (Yes=1, No=0) |
| `in.electric_vehicle_charger` | ordinal (None=0, Level 1=1, Level 2=2) |
| `in.electric_vehicle_charge_at_home` | midpoint % → float |
| `in.electric_vehicle_miles_traveled` | numérique direct |
| `in.electric_vehicle_battery` | décomposé → `ev_vehicle_type` (OHE) + `ev_battery_range_miles` (numérique) |

In [ ]:
import re

# ── Binaires ──────────────────────────────────────────────────────────────────
for col in ['in.electric_vehicle_ownership', 'in.electric_vehicle_outlet_access']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0}).astype('Int8')
        print(f'{col.replace("in.", "")} → binaire OK  {df_enc[col].value_counts().to_dict()}')

# ── Charger → ordinal (puissance croissante) ──────────────────────────────────
CHARGER_MAP = {'None': 0, 'Level 1 charger': 1, 'Level 2 charger': 2}
col = 'in.electric_vehicle_charger'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(CHARGER_MAP).astype('Int8')
    print(f'electric_vehicle_charger → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Charge at home → midpoint % float ────────────────────────────────────────
def parse_charge_pct(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s == '100%': return 1.0
    m = re.match(r'(\d+)-(\d+)%', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2 / 100
    return np.nan

col = 'in.electric_vehicle_charge_at_home'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].apply(parse_charge_pct).astype('float32')
    print(f'electric_vehicle_charge_at_home → % float OK  {sorted(df_enc[col].dropna().unique())}')

# ── Miles traveled → numérique annuel ────────────────────────────────────────
# Valeurs : 1000 – 22500 miles/an (moyenne US ≈ 13 500 miles/an)
col = 'in.electric_vehicle_miles_traveled'
if col in df_enc.columns:
    df_enc[col] = pd.to_numeric(df_enc[col], errors='coerce').astype('float32')
    print(f'electric_vehicle_miles_traveled → miles/an OK  {sorted(df_enc[col].dropna().unique())}')

# ── Battery → supprimé ───────────────────────────────────────────────────────
if 'in.electric_vehicle_battery' in df_enc.columns:
    df_enc.drop(columns=['in.electric_vehicle_battery'], inplace=True)
    print('electric_vehicle_battery supprimé')

print(f'\nShape : {df_enc.shape}')

## 10. Groupe 7 — Solaire et stockage

| Colonne | Stratégie |
|---|---|
| `in.battery` | supprimé (100% "None") |
| `in.has_pv` | binaire (Yes=1, No=0) |
| `in.pv_system_size` | numérique kWDC (`'None'` → 0) |
| `in.pv_orientation` | sin/cos circulaire (`'None'` → NaN) — 99% None, pertinent uniquement pour les 1% avec panneaux |

In [ ]:
import re

# ── Battery → suppression (100% "None") ──────────────────────────────────────
if 'in.battery' in df_enc.columns:
    df_enc.drop(columns=['in.battery'], inplace=True)
    print('battery supprimé (constante)')

# ── has_pv → binaire ──────────────────────────────────────────────────────────
col = 'in.has_pv'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0}).astype('Int8')
    n_pv = df_enc[col].sum()
    print(f'has_pv → binaire OK  ({n_pv} bâtiments avec PV, {n_pv/len(df_enc)*100:.1f}%)')

# ── PV system size → kWDC numérique ('None' → 0) ─────────────────────────────
col = 'in.pv_system_size'
if col in df_enc.columns:
    def parse_pv_size(val):
        if pd.isna(val) or str(val).strip().lower() == 'none': return 0.0
        m = re.search(r'(\d+\.?\d*)', str(val))
        return float(m.group(1)) if m else 0.0
    df_enc[col] = df_enc[col].apply(parse_pv_size).astype('float32')
    print(f'pv_system_size → kWDC OK  {sorted(df_enc[col].unique())}')

# ── PV orientation → sin/cos ('None' → NaN) ──────────────────────────────────
ORIENT_DEG = {
    'North': 0, 'Northeast': 45, 'East': 90, 'Southeast': 135,
    'South': 180, 'Southwest': 225, 'West': 270, 'Northwest': 315,
}
col = 'in.pv_orientation'
if col in df_enc.columns:
    deg = df_enc[col].map(ORIENT_DEG)   # None → NaN
    df_enc['pv_orientation_sin'] = np.sin(np.radians(deg)).astype('float32')
    df_enc['pv_orientation_cos'] = np.cos(np.radians(deg)).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('pv_orientation → sin/cos OK  (NaN pour les 99% sans panneaux)')

print(f'\nShape : {df_enc.shape}')

## Sauvegarde → `features_encodees.parquet`

In [8]:
out_path = DATA_PROCESSED / 'features_encodees.parquet'
df_enc.to_parquet(out_path, index=False)

print(f'Sauvegardé : {out_path}')
print(f'Shape final : {df_enc.shape}')

Sauvegardé : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax\data\processed\features_encodees.parquet
Shape final : (549971, 384)
